In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
from core.feature_store import FeatureStore
from core.truth import TruthVault
from core.model import Adjudicator
from core.policy import PolicyConfig
from core.backtest import run
from core.metrics import grade, StepUpModel

store = FeatureStore.load(DATA)
vault = TruthVault(DATA)
print(store)
print(f'answer key holds {len(vault):,} payments (only ~{len(store):,} of them were blocked)')

<FeatureStore 8265 cases train=6490 holdout=1775>
answer key holds 300,000 payments (only ~8,265 of them were blocked)


In [3]:
model  = Adjudicator().fit(store, vault)          # 1. learn, on train only
cfg    = PolicyConfig(cap=0.02)                    # 2. an operating point
ledger = run(store, model, cfg, 'holdout')         # 3. replay the blocked pile
out    = grade(ledger, vault, StepUpModel())       # 4. join truth, price it

print(f'{out.n_cases} appealed cases -> {out.n_released} released')
print(f'precision {out.precision:.1%}   recall of recoverable {out.recall_recoverable:.1%}')
print(f'recovered Rs {out.recovered_inr/1e7:.2f} cr')
print(f'fraud admitted Rs {out.fraud_admitted_inr/1e5:.2f} L')
print(f'net contribution Rs {out.net_contribution_inr/1e7:.2f} cr')
print(f'abstained on {out.abstention_rate:.1%} of cases')

1775 appealed cases -> 1181 released
precision 98.6%   recall of recoverable 81.1%
recovered Rs 7.04 cr
fraud admitted Rs 22.57 L
net contribution Rs 1.53 cr
abstained on 25.9% of cases
